In [5]:
# Feature engineer correlations from time series (paste into your notebook)
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

# load data (prefer in-memory tidy_all_data)
try:
    df = tidy_all_data.copy()
    print("Using in-memory `tidy_all_data`")
except NameError:
    df = pd.read_csv('../../data/processed/tidy_all_metrics.csv')
    print("Read CSV")

df['Value'] = pd.to_numeric(df['Value'], errors='coerce')

# ensure we have years: 2007, 2010, 2012-2021
years_available = sorted(df['Year'].unique())
print(f"Years available: {years_available}")

def compute_correlations(df, municipality):
    """
    Compute 7 correlations for a single municipality across its time series.
    Returns a dict with correlation values (NaN if insufficient data).
    """
    
    mun_data = df[df['Municipality_Name'] == municipality].copy()
    
    # helper: get time series as (year, value) pairs, sorted by year
    def get_ts(metric_name):
        ts = mun_data[mun_data['Metric'] == metric_name][['Year', 'Value']].drop_duplicates()
        ts = ts.sort_values('Year')
        if ts.empty:
            return np.array([]), np.array([])
        return ts['Year'].values, ts['Value'].values
    
    def safe_corr(y1, y2):
        """Pearson correlation; returns NaN if insufficient data."""
        if len(y1) < 3 or len(y2) < 3 or np.isnan(y1).all() or np.isnan(y2).all():
            return np.nan
        # align to common years
        mask = ~(np.isnan(y1) | np.isnan(y2))
        if mask.sum() < 3:
            return np.nan
        y1_clean, y2_clean = y1[mask], y2[mask]
        try:
            corr, _ = pearsonr(y1_clean, y2_clean)
            return corr
        except Exception:
            return np.nan
    
    result = {
        'Municipality_Name': municipality,
        'n_years': len(years_available),
    }
    
    # Fetch time series for all metrics we might need
    years_ref, elec_ef = get_ts('Utility_Electricity_Emissions')  # emissions factor proxy (provincial)
    _, ng_ef = get_ts('Utility_NG_Emissions')  # natural gas emissions factor
    _, res_elec_em = get_ts('Utility_Residential_Electricity_Emissions')
    _, res_polluting_em = get_ts('Utility_Residential_Polluting_Emissions')
    _, csmi_em = get_ts('Utility_CSMI_Emissions')
    _, csmi_polluting_em = get_ts('Utility_CSMI_Polluting_Emissions')
    _, csmi_elec_em = get_ts('Utility_CSMI_Electricity_Emissions')
    _, csmi_ng_em = get_ts('Utility_CSMI_NG_Emissions')
    _, total_em = get_ts('Utility_Total_Emissions')
    _, cwb_local = get_ts('CWB')
    
    # Align all to common year index
    # (use years_ref as reference, interpolate or drop where needed)
    common_years = years_ref if len(years_ref) > 0 else years_available
    
    def align_to_years(years, values, target_years):
        """Return values interpolated/resampled to target_years."""
        if len(years) == 0:
            return np.full_like(target_years, np.nan, dtype=float)
        # simple: align by exact year match, or NaN
        aligned = np.full_like(target_years, np.nan, dtype=float)
        for i, ty in enumerate(target_years):
            idx = np.where(years == ty)[0]
            if len(idx) > 0:
                aligned[i] = values[idx[0]]
        return aligned
    
    # align all series to common_years
    elec_ef_a = align_to_years(years_ref, elec_ef, common_years) if len(years_ref) > 0 else np.full_like(common_years, np.nan, dtype=float)
    ng_ef_a = align_to_years(years_ref, ng_ef, common_years) if len(years_ref) > 0 else np.full_like(common_years, np.nan, dtype=float)
    res_elec_em_a = align_to_years(years_ref, res_elec_em, common_years)
    res_polluting_em_a = align_to_years(years_ref, res_polluting_em, common_years)
    csmi_em_a = align_to_years(years_ref, csmi_em, common_years)
    csmi_polluting_em_a = align_to_years(years_ref, csmi_polluting_em, common_years)
    csmi_elec_em_a = align_to_years(years_ref, csmi_elec_em, common_years)
    csmi_ng_em_a = align_to_years(years_ref, csmi_ng_em, common_years)
    total_em_a = align_to_years(years_ref, total_em, common_years)
    cwb_a = align_to_years(years_ref, cwb_local, common_years)
    
    # --- Compute the 7 correlations ---
    
    # 1. Correlation between electricity emissions factor and residential electricity emissions
    # (high corr -> driven by emissions factor; low -> driven by consumption)
    result['corr_elec_ef_vs_res_elec_em'] = safe_corr(elec_ef_a, res_elec_em_a)
    
    # 2. Correlation between natural gas emissions factor and residential polluting emissions
    result['corr_ng_ef_vs_res_polluting_em'] = safe_corr(ng_ef_a, res_polluting_em_a)
    
    # 3. Correlation between provincial weighted average wellbeing and local CWB
    # (proxy: use total emissions as a proxy for provincial average if not available)
    # For now, correlate CWB with itself offset (not ideal; ideally you'd have provincial CWB)
    # Skip this for now or use a placeholder
    result['corr_prov_wellbeing_vs_local_cwb'] = np.nan  # requires external provincial data
    
    # 4. Correlation between total emissions percent change and residential emissions percent change
    # (compute pct change over time: (end - start) / start)
    def pct_change_ts(values):
        """Compute percent change from first to each year."""
        if len(values) < 2 or np.isnan(values[0]) or values[0] == 0:
            return np.full_like(values, np.nan)
        pct = (values - values[0]) / np.abs(values[0]) * 100
        return pct
    
    total_em_pct = pct_change_ts(total_em_a)
    res_em_pct = pct_change_ts(res_elec_em_a + res_polluting_em_a)  # total residential
    result['corr_total_em_pct_vs_res_em_pct'] = safe_corr(total_em_pct, res_em_pct)
    
    # 5. Correlation between percent changes of electricity emissions and CSMI emissions
    csmi_elec_em_pct = pct_change_ts(csmi_elec_em_a)
    csmi_em_pct = pct_change_ts(csmi_em_a)
    result['corr_csmi_elec_em_pct_vs_csmi_em_pct'] = safe_corr(csmi_elec_em_pct, csmi_em_pct)
    
    # 6. Correlation between percent changes of CSMI emissions and total emissions
    result['corr_csmi_em_pct_vs_total_em_pct'] = safe_corr(csmi_em_pct, total_em_pct)
    
    # 7. Correlation between percent changes of natural gas emissions and total emissions
    csmi_ng_em_pct = pct_change_ts(csmi_ng_em_a)
    result['corr_csmi_ng_em_pct_vs_total_em_pct'] = safe_corr(csmi_ng_em_pct, total_em_pct)
    
    # 8. Correlation between percent changes of natural gas emissions and CSMI emissions
    result['corr_csmi_ng_em_pct_vs_csmi_em_pct'] = safe_corr(csmi_ng_em_pct, csmi_em_pct)
    
    return result

# Compute for all municipalities
municipalities = df['Municipality_Name'].unique()
print(f"Computing correlations for {len(municipalities)} municipalities...")

corr_results = []
for i, mun in enumerate(municipalities):
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(municipalities)}")
    row = compute_correlations(df, mun)
    corr_results.append(row)

# assemble into dataframe
tidy_correlations = pd.DataFrame(corr_results)

print(f"\nComputed {len(tidy_correlations)} municipality correlation rows")
print("Sample correlations:")
display(tidy_correlations.head())

# save to CSV
out_path = '../../data/processed/municipality_correlations_features.csv'
tidy_correlations.to_csv(out_path, index=False)
print(f"Saved to: {out_path}")

# summary stats per correlation
print("\nSummary of correlation features:")
corr_cols = [c for c in tidy_correlations.columns if c.startswith('corr_')]
display(tidy_correlations[corr_cols].describe().round(3))

Read CSV
Years available: [np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]
Computing correlations for 148 municipalities...
  50/148
  100/148

Computed 148 municipality correlation rows
Sample correlations:


,Municipality_Name,n_years,corr_elec_ef_vs_res_elec_em,corr_ng_ef_vs_res_polluting_em,corr_prov_wellbeing_vs_local_cwb,corr_total_em_pct_vs_res_em_pct,corr_csmi_elec_em_pct_vs_csmi_em_pct,corr_csmi_em_pct_vs_total_em_pct,corr_csmi_ng_em_pct_vs_total_em_pct,corr_csmi_ng_em_pct_vs_csmi_em_pct
0,Abbotsford,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Alert Bay,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Anmore,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Armstrong,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Ashcroft,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Saved to: ../../data/processed/municipality_correlations_features.csv

Summary of correlation features:


,corr_elec_ef_vs_res_elec_em,corr_ng_ef_vs_res_polluting_em,corr_prov_wellbeing_vs_local_cwb,corr_total_em_pct_vs_res_em_pct,corr_csmi_elec_em_pct_vs_csmi_em_pct,corr_csmi_em_pct_vs_total_em_pct,corr_csmi_ng_em_pct_vs_total_em_pct,corr_csmi_ng_em_pct_vs_csmi_em_pct
count,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
